## 1. Análisis exploratorio (EDA) - Customer Churn

En este notebook hacemos el análisis exploratorio del dataset que nos dio la materia
para la elaboración del proyecto. La idea es entender cómo viene
la información antes de armar el pipeline: ver qué tipo de datos tenemos, si hay
columnas con datos faltantes, cómo está repartida la variable que queremos predecir
(Churn) y si aparece algo raro que después tengamos que resolver.

In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/customer_churn_historical.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,6684-LWVH,Male,0,No,No,4,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,One year,No,Mailed check,27.56,107.47,No
1,7027-JIFO,Male,0,No,No,35,Yes,No,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Month-to-month,Yes,Mailed check,30.52,1020.47,No
2,5981-VOQO,Female,0,No,No,45,Yes,Yes,DSL,No,...,No,No,Yes,No,Month-to-month,Yes,Bank transfer (automatic),66.87,2686.35,No
3,7266-HYDM,Male,0,Yes,No,31,Yes,Yes,Fiber optic,No,...,No,No,No,Yes,Month-to-month,Yes,Bank transfer (automatic),86.95,2536.96,Yes
4,2821-JDLS,Female,0,Yes,No,5,Yes,Yes,No,No internet service,...,No internet service,No internet service,No internet service,No internet service,Two year,No,Mailed check,30.18,139.12,No


## 2. Cómo viene el dataset

Primero miramos cuántas filas y columnas tiene el archivo, y qué tipo de dato
guarda cada columna (si es número o texto). También nos sirve para ver si alguna
columna tiene menos datos que las demás, lo que ya nos señala dónde
hay valores faltantes.

In [2]:
print("Dimensiones (filas, columnas):", df.shape)
print()
df.info()

Dimensiones (filas, columnas): (7043, 21)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16 

## 3. Valores faltantes

El README del dataset advierte que `TotalCharges` tiene faltantes intencionales.
Asique lo que hacemos es confirmar cuántos nulos hay por columna.

In [3]:
df.isnull().sum().sort_values(ascending=False)

TotalCharges        26
gender               0
SeniorCitizen        0
Partner              0
customerID           0
Dependents           0
tenure               0
MultipleLines        0
PhoneService         0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
InternetService      0
TechSupport          0
StreamingTV          0
Contract             0
StreamingMovies      0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
Churn                0
dtype: int64

## 4. Cómo está repartido el Churn

Una vez que confirmamos los faltantes, pasamos a mirar la variable que
queremos predecir. Contamos cuántos clientes se quedaron y cuántos se
fueron, y también calculamos el porcentaje sobre el total, para saber
si el dataset está balanceado o no. Esto nos importa porque si hay
muchos más clientes que se quedan que clientes que se van, no podemos
guiarnos solo por el Accuracy para decir si un modelo es bueno.

In [8]:
conteo = df["Churn"].value_counts()
porcentaje = df["Churn"].value_counts(normalize=True) * 100

print(conteo)
print()
print(porcentaje)

Churn
No     5186
Yes    1857
Name: count, dtype: int64

Churn
No     73.633395
Yes    26.366605
Name: proportion, dtype: float64


## 5. Cómo está repartido el Churn

Una vez que confirmamos los faltantes, pasamos a mirar la variable que
queremos predecir. Contamos cuántos clientes se quedaron y cuántos se
fueron, para saber si el dataset está balanceado o no. Esto nos importa
porque si hay muchos más clientes que se quedan que clientes que se van,
no podemos guiarnos solo por el Accuracy para decir si un modelo es bueno.

In [6]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges
count,7043.000000,7043.000000,7043.000000,7017.000000
mean,0.170098,35.168394,68.168503,2312.077586
std,0.375746,18.901478,24.980659,1573.967858
min,0.000000,0.000000,18.000000,0.000000
25%,0.000000,20.000000,55.360000,986.630000
50%,0.000000,35.000000,73.910000,2013.200000
75%,0.000000,51.000000,88.000000,3452.850000
max,1.000000,72.000000,114.410000,7761.340000


## 6. Variables categóricas

Para cerrar el análisis, revisamos también las columnas de texto.
Miramos qué categorías tiene cada una y cuántos clientes caen en cada
una, para chequear que no haya errores de tipeo o categorías repetidas
escritas de forma distinta.

In [7]:
for col in df.select_dtypes(include="object").columns:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- customerID ---
customerID
6684-LWVH    1
7027-JIFO    1
5981-VOQO    1
7266-HYDM    1
2821-JDLS    1
            ..
5878-LZXJ    1
8490-ENJV    1
7152-RWIV    1
5768-FYEE    1
1398-XMGM    1
Name: count, Length: 7043, dtype: int64

--- gender ---
gender
Male      3560
Female    3483
Name: count, dtype: int64

--- Partner ---
Partner
No     3623
Yes    3420
Name: count, dtype: int64

--- Dependents ---
Dependents
No     4954
Yes    2089
Name: count, dtype: int64

--- PhoneService ---
PhoneService
Yes    6368
No      675
Name: count, dtype: int64

--- MultipleLines ---
MultipleLines
No                  3456
Yes                 2912
No phone service     675
Name: count, dtype: int64

--- InternetService ---
InternetService
Fiber optic    3165
DSL            2329
No             1549
Name: count, dtype: int64

--- OnlineSecurity ---
OnlineSecurity
No                     3203
Yes                    2291
No internet service    1549
Name: count, dtype: int64

--- OnlineBackup ---
OnlineBac